# Week 8 — Module 19: Function-specific optimisation and surrogate validation

This notebook follows the Week 7 cumulative-data workflow. It accepts the existing `updated_data_week_7` directory or the module's `Data` directory, and safely handles Week 7 observations that are already included in the input CSVs. It preserves the original Week 8 modelling settings and final candidate coordinates.

## Load observations

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Use the Week 7 datasets as the starting point for Week 8.
search_roots = [Path.cwd(), *Path.cwd().parents]
relative_locations = [
    Path("7. Module 18/updated_data_week_7"),
    Path("updated_data_week_7"),
    Path("8. Module 19/Data"),
    Path("Data"),
]
data_folder = next(
    (base / location for base in search_roots for location in relative_locations
     if all((base / location / f"function_{i}.csv").is_file() for i in range(1, 9))),
    None,
)
if data_folder is None:
    raise FileNotFoundError("Could not find all eight Week 7 function CSVs. Place them in updated_data_week_7 or Data.")

data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}
print(f"Loaded eight datasets from {data_folder}")

Loaded eight datasets from C:\Users\charl\OneDrive - thesuperyachtgroup.com\Desktop\Imperial AI course\7. Capstone Project\8. Module 19\Data


## Week 7 observations

In [2]:
# Week 7 submitted points
new_points = {
    1: [0.858598, 0.697368],
    2: [0.706079, 0.158030],
    3: [0.450320, 0.561567, 0.490526],
    4: [0.384332, 0.391111, 0.336887, 0.416488],
    5: [1.000000, 1.000000, 1.000000, 1.000000],
    6: [0.387946, 0.435172, 0.345126, 0.743513, 0.076825],
    7: [0.054205, 0.216830, 0.421532, 0.346756, 0.385001, 0.543902],
    8: [0.030327, 0.231267, 0.050266, 0.208295,
        0.818520, 0.414744, 0.307096, 0.583908]
}

# Returned outputs
new_outputs = {
    1: 1.539769610658716e-40,
    2: 0.5699038779230302,
    3: -0.010189939911026829,
    4: 0.18691172535641742,
    5: 8662.4825,
    6: -0.3886868054486413,
    7: 2.369288578893768,
    8: 9.9308110991456
}

In [3]:
# Some cumulative input CSVs already contain the Week 7 result.
# Compare against the historical observations, without counting that result as the previous best.
week7_results = []
for i in range(1, 9):
    df = data[i]
    input_columns = [col for col in df.columns if col != "output"]
    point = np.asarray(new_points[i], dtype=float)
    matching = np.all(np.isclose(df[input_columns].to_numpy(dtype=float), point, atol=5e-7, rtol=0), axis=1)
    historical = df.loc[~matching, "output"]
    if historical.empty:
        raise ValueError(f"F{i}: no historical observations available for comparison")
    previous_best = float(historical.max())
    new_output = float(new_outputs[i])
    week7_results.append({"Function": i, "Previous best": previous_best,
                          "New output": new_output, "Updated best": max(previous_best, new_output),
                          "Improved": new_output > previous_best})
from IPython.display import display
display(pd.DataFrame(week7_results).round(6))

,Function,Previous best,New output,Updated best,Improved
0,1,0.000000,0.000000,0.000000,False
1,2,0.746962,0.569904,0.746962,False
2,3,-0.015498,-0.010190,-0.010190,True
3,4,-0.200431,0.186912,0.186912,True
4,5,7121.151955,8662.482500,8662.482500,True
5,6,-0.379740,-0.388687,-0.379740,False
6,7,2.929370,2.369289,2.929370,False
7,8,9.911249,9.930811,9.930811,True


In [4]:
for i in range(1, 9):
    input_columns = [col for col in data[i].columns if col != "output"]
    point = np.asarray(new_points[i], dtype=float)
    existing = data[i][input_columns].to_numpy(dtype=float)
    matching = np.flatnonzero(np.all(np.isclose(existing, point, atol=5e-7, rtol=0), axis=1))
    if len(matching):
        recorded = float(data[i].iloc[matching[0]]["output"])
        if not np.isclose(recorded, new_outputs[i], atol=1e-6, rtol=1e-8):
            raise ValueError(f"F{i}: existing Week 7 point has a different output")
    else:
        row = dict(zip(input_columns, point))
        row["output"] = new_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([row])], ignore_index=True)
print("Observations:", ", ".join(f"F{i}: {len(data[i])}" for i in range(1, 9)))

Observations: F1: 17, F2: 17, F3: 22, F4: 37, F5: 27, F6: 27, F7: 37, F8: 47


## Gaussian Process models

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
gp_models = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values
    y = data[i]['output'].values
    gp = GaussianProcessRegressor(kernel=RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0)), normalize_y=True, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    gp_models[i] = gp

print("GP length scales:", ", ".join(f"F{i}: {gp_models[i].kernel_.length_scale:.4g}" for i in range(1, 9)))

GP length scales: F1: 1e-05, F2: 1e-05, F3: 0.1995, F4: 0.4159, F5: 0.2731, F6: 0.5809, F7: 0.3021, F8: 1.036


## Prediction diagnostics

In [6]:
rng = np.random.default_rng(42)
candidate_sets = {}
for i in range(1, 9):
    n_dims = data[i].shape[1] - 1
    candidates = rng.random((100000, n_dims))
    candidate_sets[i] = candidates
    pred_mean, pred_std = gp_models[i].predict(candidates, return_std=True)

## Neural-network ensembles

**Architecture note:** The original Week 8 implementation uses two 20-neuron hidden layers (as in Week 7), rather than the 16-neuron layers used in earlier rounds. This is preserved to avoid changing results.

In [7]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
nn_models = {}
nn_x_scalers = {}
nn_y_scalers = {}
nn_training_losses = {}
n_ensemble = 5
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float).reshape(-1, 1)
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y).ravel()
    models = []
    losses = []
    for seed in range(n_ensemble):
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=seed)
        model.fit(X_scaled, y_scaled)
        models.append(model)
        losses.append(model.loss_)
    nn_models[i] = models
    nn_x_scalers[i] = x_scaler
    nn_y_scalers[i] = y_scaler
    nn_training_losses[i] = losses

print("NN training losses:", ", ".join(f"F{i}: {np.mean(nn_training_losses[i]):.6f}" for i in range(1, 9)))

NN training losses: F1: 0.000963, F2: 0.010989, F3: 0.000416, F4: 0.000284, F5: 0.000230, F6: 0.000370, F7: 0.000338, F8: 0.000381


## Neural-network LOOCV

In [8]:
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import numpy as np
nn_loocv_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    nn_predictions = []
    baseline_predictions = []
    actual_values = []
    for j in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[j] = False
        X_train = X[mask]
        y_train = y[mask]
        X_test = X[j:j + 1]
        y_test = y[j]
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=42)
        model.fit(X_train_scaled, y_train_scaled)
        pred_scaled = model.predict(X_test_scaled)[0]
        pred = y_scaler.inverse_transform([[pred_scaled]])[0, 0]
        nn_predictions.append(pred)
        baseline_predictions.append(np.mean(y_train))
        actual_values.append(y_test)
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_loocv_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

display(pd.DataFrame([{"Function": i, **nn_loocv_results[i]} for i in range(1, 9)]).round(6))

,Function,nn_mae,baseline_mae,beats_baseline
0,1,0.001704,0.001050,False
1,2,0.219676,0.237993,True
2,3,0.061985,0.051662,False
3,4,1.825570,7.284354,True
4,5,230.163312,2049.255132,True
5,6,0.241315,0.513894,True
6,7,0.399610,0.639930,True
7,8,0.354980,0.962395,True


## GP acquisition candidates

In [9]:
from scipy.stats import norm
from scipy.spatial.distance import cdist
import numpy as np
kappa_values = [0.5, 1.5]
xi_percentages = [0.0, 0.05]
gp_candidate_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    candidates = candidate_sets[i]
    mean, std = gp_models[i].predict(candidates, return_std=True)
    current_best = np.max(y)
    y_std = np.std(y)
    results = {'UCB': {}, 'EI': {}}
    for kappa in kappa_values:
        ucb = mean + kappa * std
        idx = np.argmax(ucb)
        point = candidates[idx]
        nearest_distance = np.min(cdist(point.reshape(1, -1), X))
        results['UCB'][kappa] = {'point': point, 'predicted': mean[idx], 'uncertainty': std[idx], 'score': ucb[idx], 'nearest_distance': nearest_distance}
    for xi_pct in xi_percentages:
        xi = xi_pct * y_std
        improvement = mean - current_best - xi
        with np.errstate(divide='ignore', invalid='ignore'):
            Z = np.divide(improvement, std, out=np.zeros_like(improvement), where=std > 0)
        ei = improvement * norm.cdf(Z) + std * norm.pdf(Z)
        ei[std == 0] = 0
        idx = np.argmax(ei)
        point = candidates[idx]
        nearest_distance = np.min(cdist(point.reshape(1, -1), X))
        results['EI'][xi_pct] = {'point': point, 'predicted': mean[idx], 'uncertainty': std[idx], 'score': ei[idx], 'nearest_distance': nearest_distance}
    gp_candidate_results[i] = results

# Display computed results
rows = []
for i, groups in gp_candidate_results.items():
    for method, settings in groups.items():
        for setting, r in settings.items():
            rows.append({'Function': f'F{i}', 'Method': method, 'Setting': f'{setting:g}', 'Candidate': np.array2string(r['point'], precision=6, separator=', '), 'GP mean': r['predicted'], 'GP std': r['uncertainty'], 'Acquisition': r['score'], 'Nearest distance': r['nearest_distance']})
display(pd.DataFrame(rows).round(6).style.format(precision=6))

,Function,Method,Setting,Candidate,GP mean,GP std,Acquisition,Nearest distance
0,F1,UCB,0.5,"[0.094177, 0.975622]",-0.000560,0.001584,0.000232,0.095343
1,F1,UCB,1.5,"[0.094177, 0.975622]",-0.000560,0.001584,0.001816,0.095343
2,F1,EI,0,"[0.094177, 0.975622]",-0.000560,0.001584,0.000391,0.095343
3,F1,EI,0.05,"[0.094177, 0.975622]",-0.000560,0.001584,0.000363,0.095343
4,F2,UCB,0.5,"[0.706079, 0.15803 ]",0.569650,0.011502,0.575400,0.000000
5,F2,UCB,1.5,"[0.403694, 0.391789]",0.319732,0.255139,0.702440,0.113423
6,F2,EI,0,"[0.403694, 0.391789]",0.319732,0.255139,0.004964,0.113423
7,F2,EI,0.05,"[0.403694, 0.391789]",0.319732,0.255139,0.004394,0.113423
8,F3,UCB,0.5,"[0.463879, 0.497888, 0.452731]",-0.012395,0.013193,-0.005798,0.075282
9,F3,UCB,1.5,"[0.779394, 0.644459, 0.001936]",-0.050055,0.059019,0.038474,0.206813


## Neural-network candidates

In [10]:
from scipy.spatial.distance import cdist
nn_candidate_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    candidates = candidate_sets[i]
    candidates_scaled = nn_x_scalers[i].transform(candidates)
    model_predictions = []
    for model in nn_models[i]:
        pred_scaled = model.predict(candidates_scaled)
        pred = nn_y_scalers[i].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        model_predictions.append(pred)
    model_predictions = np.array(model_predictions)
    mean_prediction = np.mean(model_predictions, axis=0)
    model_disagreement = np.std(model_predictions, axis=0)
    idx = np.argmax(mean_prediction)
    selected_point = candidates[idx]
    selected_prediction = mean_prediction[idx]
    selected_disagreement = model_disagreement[idx]
    best_idx = np.argmax(y)
    current_best_point = X[best_idx]
    distance_from_best = np.linalg.norm(selected_point - current_best_point)
    nearest_distance = np.min(cdist(selected_point.reshape(1, -1), X))
    nn_candidate_results[i] = {'point': selected_point, 'predicted': selected_prediction, 'disagreement': selected_disagreement, 'distance_from_best': distance_from_best, 'nearest_distance': nearest_distance}

# Display computed results
display(pd.DataFrame([{'Function': f'F{i}', 'NN candidate': np.array2string(r['point'], precision=6, separator=', '), 'NN prediction': r['predicted'], 'Ensemble disagreement': r['disagreement'], 'Distance from best': r['distance_from_best'], 'Nearest distance': r['nearest_distance']} for i, r in nn_candidate_results.items()]).round(6))

,Function,NN candidate,NN prediction,Ensemble disagreement,Distance from best,Nearest distance
0,F1,"[0.724967, 0.999094]",0.001008,0.000982,0.266163,0.144154
1,F2,"[0.697103, 0.994806]",0.696942,0.049182,0.856800,0.068466
2,F3,"[0.437451, 0.559044, 0.487388]",-0.009593,0.001325,0.013484,0.013484
3,F4,"[0.384332, 0.391111, 0.336887, 0.416488]",0.186593,0.013201,0.000001,0.000001
4,F5,"[0.995347, 0.980915, 0.954873, 0.970449]",7502.790594,117.374871,0.057407,0.057407
5,F6,"[0.434907, 0.336577, 0.469677, 0.823758, 0.051...",-0.345084,0.039197,0.173710,0.173710
6,F7,"[0.174885, 0.324058, 0.479146, 0.269144, 0.331...",2.928474,0.000686,0.000001,0.000001
7,F8,"[0.007931, 0.186803, 0.075203, 0.062718, 0.632...",10.207430,0.119102,0.447995,0.447995


## GP–NN comparison

In [11]:
gp_nn_distances = {}
for i in range(1, 9):
    nn_point = nn_candidate_results[i]['point']
    distances = {'UCB k=0.5': np.linalg.norm(nn_point - gp_candidate_results[i]['UCB'][0.5]['point']), 'UCB k=1.5': np.linalg.norm(nn_point - gp_candidate_results[i]['UCB'][1.5]['point']), 'EI xi=0%': np.linalg.norm(nn_point - gp_candidate_results[i]['EI'][0.0]['point']), 'EI xi=5%': np.linalg.norm(nn_point - gp_candidate_results[i]['EI'][0.05]['point'])}
    gp_nn_distances[i] = distances

# Display computed results
display(pd.DataFrame([{'Function': f'F{i}', **distances} for i, distances in gp_nn_distances.items()]).round(6))

,Function,UCB k=0.5,UCB k=1.5,EI xi=0%,EI xi=5%
0,F1,0.631226,0.631226,0.631226,0.631226
1,F2,0.836824,0.670611,0.670611,0.670611
2,F3,0.075098,0.599904,0.579013,0.599904
3,F4,0.041900,0.059061,0.925379,0.925379
4,F5,0.000000,0.000000,1.368297,1.368297
5,F6,0.091968,0.213678,0.176242,0.213678
6,F7,0.000000,0.222032,0.222032,0.222032
7,F8,0.339623,1.005327,1.073629,1.073629


## Neighbourhood evidence

In [12]:
neighbourhood_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    candidates_to_check = {'UCB k=0.5': gp_candidate_results[i]['UCB'][0.5]['point'], 'UCB k=1.5': gp_candidate_results[i]['UCB'][1.5]['point'], 'EI xi=0%': gp_candidate_results[i]['EI'][0.0]['point'], 'EI xi=5%': gp_candidate_results[i]['EI'][0.05]['point'], 'NN': nn_candidate_results[i]['point']}
    function_results = {}
    for method, point in candidates_to_check.items():
        distances = cdist(point.reshape(1, -1), X).ravel()
        nearest_idx = np.argsort(distances)[:3]
        nearest_outputs = y[nearest_idx]
        nearest_distances = distances[nearest_idx]
        function_results[method] = {'point': point, 'nearest_indices': nearest_idx, 'nearest_outputs': nearest_outputs, 'nearest_distances': nearest_distances}
    neighbourhood_results[i] = function_results

# Display computed results
rows = []
for i, methods in neighbourhood_results.items():
    for method, r in methods.items():
        rows.append({'Function': f'F{i}', 'Method': method, 'Candidate': np.array2string(r['point'], precision=6, separator=', '), 'Nearest outputs': np.array2string(r['nearest_outputs'], precision=6, separator=', '), 'Nearest distances': np.array2string(r['nearest_distances'], precision=6, separator=', ')})
display(pd.DataFrame(rows))

,Function,Method,Candidate,Nearest outputs,Nearest distances
0,F1,UCB k=0.5,"[0.094177, 0.975622]","[0.000000e+00, 1.322677e-79, 1.033078e-46]","[0.095343, 0.309762, 0.489601]"
1,F1,UCB k=1.5,"[0.094177, 0.975622]","[0.000000e+00, 1.322677e-79, 1.033078e-46]","[0.095343, 0.309762, 0.489601]"
2,F1,EI xi=0%,"[0.094177, 0.975622]","[0.000000e+00, 1.322677e-79, 1.033078e-46]","[0.095343, 0.309762, 0.489601]"
3,F1,EI xi=5%,"[0.094177, 0.975622]","[0.000000e+00, 1.322677e-79, 1.033078e-46]","[0.095343, 0.309762, 0.489601]"
4,F1,NN,"[0.724967, 0.999094]","[2.535001e-40, 1.033078e-46, 7.710875e-16]","[0.144154, 0.192092, 0.266163]"
5,F2,UCB k=0.5,"[0.706079, 0.15803 ]","[0.569904, 0.746962, 0.471844]","[4.510272e-07, 2.107597e-02, 5.165265e-02]"
6,F2,UCB k=1.5,"[0.403694, 0.391789]","[ 0.214965, -0.013858, -0.065624]","[0.113423, 0.189439, 0.264478]"
7,F2,EI xi=0%,"[0.403694, 0.391789]","[ 0.214965, -0.013858, -0.065624]","[0.113423, 0.189439, 0.264478]"
8,F2,EI xi=5%,"[0.403694, 0.391789]","[ 0.214965, -0.013858, -0.065624]","[0.113423, 0.189439, 0.264478]"
9,F2,NN,"[0.697103, 0.994806]","[0.611205, 0.36601 , 0.507418]","[0.068466, 0.077214, 0.086102]"


## Function 1: alternative kernels

In [13]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.metrics import mean_absolute_error
import numpy as np
X = data[1].drop(columns='output').values.astype(float)
y = data[1]['output'].values.astype(float)
kernels = {'RBF': RBF(length_scale=0.1, length_scale_bounds=(1e-05, 100.0)), 'Matern 3/2': Matern(length_scale=0.1, length_scale_bounds=(1e-05, 100.0), nu=1.5), 'Matern 5/2': Matern(length_scale=0.1, length_scale_bounds=(1e-05, 100.0), nu=2.5), 'RBF + noise': RBF(length_scale=0.1, length_scale_bounds=(1e-05, 100.0)) + WhiteKernel(noise_level=1e-05, noise_level_bounds=(1e-08, 1.0))}
kernel_rows = []
for name, kernel in kernels.items():
    predictions = []
    actual = []
    baseline_predictions = []
    for j in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[j] = False
        X_train = X[mask]
        y_train = y[mask]
        X_test = X[j:j + 1]
        y_test = y[j]
        gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
        gp.fit(X_train, y_train)
        pred = gp.predict(X_test)[0]
        predictions.append(pred)
        actual.append(y_test)
        baseline_predictions.append(np.mean(y_train))
    gp_mae = mean_absolute_error(actual, predictions)
    baseline_mae = mean_absolute_error(actual, baseline_predictions)
    final_gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
    final_gp.fit(X, y)

# Display computed results
    kernel_rows.append({'Kernel': name, 'LOOCV MAE': gp_mae, 'Baseline MAE': baseline_mae, 'Beats baseline': gp_mae < baseline_mae, 'Fitted kernel': str(final_gp.kernel_)})
display(pd.DataFrame(kernel_rows).round(6))

,Kernel,LOOCV MAE,Baseline MAE,Beats baseline,Fitted kernel
0,RBF,0.001099,0.00105,False,RBF(length_scale=0.00409)
1,Matern 3/2,0.001106,0.00105,False,"Matern(length_scale=0.00419, nu=1.5)"
2,Matern 5/2,0.001107,0.00105,False,"Matern(length_scale=0.00419, nu=2.5)"
3,RBF + noise,0.001099,0.00105,False,RBF(length_scale=0.000124) + WhiteKernel(noise...


## Function 1: space-filling search

In [14]:
X_f1 = data[1].drop(columns='output').values.astype(float)
rng = np.random.default_rng(42)
f1_exploration_candidates = rng.random((100000, 2))
distances = cdist(f1_exploration_candidates, X_f1)
nearest_distances = np.min(distances, axis=1)
order = np.argsort(nearest_distances)[::-1]
selected = []
for idx in order:
    point = f1_exploration_candidates[idx]
    if len(selected) == 0:
        selected.append(idx)
    else:
        existing_selected = f1_exploration_candidates[selected]
        if np.min(cdist(point.reshape(1, -1), existing_selected)) > 0.1:
            selected.append(idx)
    if len(selected) == 5:
        break
for rank, idx in enumerate(selected, start=1):
    point = f1_exploration_candidates[idx]

# Display computed results
f1_space_results = [{'Rank': rank, 'Candidate': np.array2string(f1_exploration_candidates[idx], precision=6, separator=', '), 'Nearest distance': nearest_distances[idx]} for rank, idx in enumerate(selected, start=1)]
display(pd.DataFrame(f1_space_results).round(6))

,Rank,Candidate,Nearest distance
0,1,"[9.982943e-01, 7.088347e-04]",0.307658
1,2,"[0.684654, 0.001334]",0.305975
2,3,"[0.006886, 0.695109]",0.301267
3,4,"[0.786713, 0.000852]",0.269276
4,5,"[0.898148, 0.005297]",0.265794


## Function 1: candidate comparison

In [15]:
f1_top_candidates = np.array([[0.998294, 0.000709], [0.684654, 0.001334], [0.006886, 0.695109]])
X = data[1].drop(columns='output').values.astype(float)
y = data[1]['output'].values.astype(float)
gp = gp_models[1]
candidates_scaled = nn_x_scalers[1].transform(f1_top_candidates)
nn_preds = []
for model in nn_models[1]:
    pred_scaled = model.predict(candidates_scaled)
    pred = nn_y_scalers[1].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    nn_preds.append(pred)
nn_preds = np.array(nn_preds)
nn_mean = np.mean(nn_preds, axis=0)
nn_disagreement = np.std(nn_preds, axis=0)
gp_mean, gp_std = gp.predict(f1_top_candidates, return_std=True)
f1_comparison_rows = []
for j, point in enumerate(f1_top_candidates):
    distances = cdist(point.reshape(1, -1), X).ravel()
    nearest_idx = np.argsort(distances)[:3]

# Display computed results
    f1_comparison_rows.append({'Candidate': np.array2string(point, precision=6, separator=', '), 'Nearest distance': distances[nearest_idx[0]], 'Nearest outputs': np.array2string(y[nearest_idx], precision=6, separator=', '), 'GP prediction': gp_mean[j], 'GP uncertainty': gp_std[j], 'NN prediction': nn_mean[j], 'NN disagreement': nn_disagreement[j]})
display(pd.DataFrame(f1_comparison_rows).round(6))

,Candidate,Nearest distance,Nearest outputs,GP prediction,GP uncertainty,NN prediction,NN disagreement
0,"[9.98294e-01, 7.09000e-04]",0.307658,"[ 3.341771e-124, -2.446767e-120, 9.656223e-041]",-0.00056,0.001584,0.000593,0.001163
1,"[0.684654, 0.001334]",0.305975,"[ 3.341771e-124, -2.159249e-054, -2.089093e-091]",-0.00056,0.001584,0.000432,0.000420
2,"[0.006886, 0.695109]",0.301267,"[3.606771e-81, 0.000000e+00, 1.322677e-79]",-0.00056,0.001584,-0.000247,0.000373


## Local refinement

In [16]:
from scipy.spatial.distance import cdist
import numpy as np
rng = np.random.default_rng(42)
local_results = {}
local_display_rows = []
for i in [4, 5, 7]:
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_output = y[best_idx]
    n_candidates = 100000
    local_scale = 0.08
    candidates = best_point + rng.normal(0, local_scale, size=(n_candidates, X.shape[1]))
    candidates = np.clip(candidates, 0, 1)
    distances = cdist(candidates, X)
    nearest_distance = np.min(distances, axis=1)
    valid = nearest_distance > 1e-06
    candidates = candidates[valid]
    nearest_distance = nearest_distance[valid]
    gp_mean, gp_std = gp_models[i].predict(candidates, return_std=True)
    candidates_scaled = nn_x_scalers[i].transform(candidates)
    nn_predictions = []
    for model in nn_models[i]:
        pred_scaled = model.predict(candidates_scaled)
        pred = nn_y_scalers[i].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        nn_predictions.append(pred)
    nn_predictions = np.array(nn_predictions)
    nn_mean = np.mean(nn_predictions, axis=0)
    nn_disagreement = np.std(nn_predictions, axis=0)
    gp_z = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)
    nn_z = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
    combined_score = (gp_z + nn_z) / 2
    best_candidate_idx = np.argmax(combined_score)
    selected = candidates[best_candidate_idx]
    local_results[i] = selected

# Display computed results
    local_display_rows.append({'Function': f'F{i}', 'Candidate': np.array2string(selected, precision=6, separator=', '), 'GP prediction': gp_mean[best_candidate_idx], 'NN prediction': nn_mean[best_candidate_idx], 'NN disagreement': nn_disagreement[best_candidate_idx], 'Nearest distance': nearest_distance[best_candidate_idx], 'Previous best': best_output})
display(pd.DataFrame(local_display_rows).round(6))

,Function,Candidate,GP prediction,NN prediction,NN disagreement,Nearest distance,Previous best
0,F4,"[0.386248, 0.38101 , 0.343907, 0.423092]",0.201383,0.280578,0.053296,0.014092,0.186912
1,F5,"[1. , 1. , 0.999995, 1. ]",8662.440815,8657.455798,4.375012,0.000005,8662.482500
2,F7,"[0.157801, 0.305871, 0.468629, 0.230835, 0.339...",2.920647,2.936110,0.073402,0.047854,2.929370


## Function 5 novelty constraint

In [17]:
i = 5
X = data[i].drop(columns='output').values.astype(float)
y = data[i]['output'].values.astype(float)
best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]
rng = np.random.default_rng(42)
n_candidates = 100000
local_scale = 0.08
candidates = best_point + rng.normal(0, local_scale, size=(n_candidates, X.shape[1]))
candidates = np.clip(candidates, 0, 1)
distances = cdist(candidates, X)
nearest_distance = np.min(distances, axis=1)
min_novelty = 0.03
valid = nearest_distance >= min_novelty
candidates = candidates[valid]
nearest_distance = nearest_distance[valid]
gp_mean, gp_std = gp_models[i].predict(candidates, return_std=True)
candidates_scaled = nn_x_scalers[i].transform(candidates)
nn_predictions = []
for model in nn_models[i]:
    pred_scaled = model.predict(candidates_scaled)
    pred = nn_y_scalers[i].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    nn_predictions.append(pred)
nn_predictions = np.array(nn_predictions)
nn_mean = np.mean(nn_predictions, axis=0)
nn_disagreement = np.std(nn_predictions, axis=0)
gp_z = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)
nn_z = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
combined_score = (gp_z + nn_z) / 2
best_candidate_idx = np.argmax(combined_score)
selected = candidates[best_candidate_idx]

# Display computed results
display(pd.DataFrame([{'Function': 'F5', 'Candidate': np.array2string(selected, precision=6, separator=', '), 'GP prediction': gp_mean[best_candidate_idx], 'NN prediction': nn_mean[best_candidate_idx], 'NN disagreement': nn_disagreement[best_candidate_idx], 'Nearest distance': nearest_distance[best_candidate_idx], 'Minimum novelty': min_novelty, 'Previous best': best_output}]).round(6))

,Function,Candidate,GP prediction,NN prediction,NN disagreement,Nearest distance,Minimum novelty,Previous best
0,F5,"[1. , 1. , 0.97, 1. ]",8391.490608,8342.417329,66.559904,0.03,0.03,8662.4825


## Final validation and submission

In [18]:
final_candidates = {1: np.array([0.998294, 0.000709]), 2: np.array([0.697103, 0.994806]), 3: np.array([0.463879, 0.497888, 0.452731]), 4: np.array([0.386248, 0.38101, 0.343907, 0.423092]), 5: np.array([1.0, 1.0, 0.97, 1.0]), 6: np.array([0.426973, 0.384648, 0.468006, 0.752822, 0.084308]), 7: np.array([0.157801, 0.305871, 0.468629, 0.230835, 0.339422, 0.619025]), 8: np.array([0.026226, 0.080912, 0.12825, 0.056836, 0.74825, 0.399531, 0.253527, 0.72075])}
all_valid = True
for i in range(1, 9):
    candidate = final_candidates[i]
    X = data[i].drop(columns='output').values.astype(float)
    correct_dimensions = len(candidate) == X.shape[1]
    within_bounds = np.all((candidate >= 0) & (candidate <= 1))
    distances = cdist(candidate.reshape(1, -1), X).ravel()
    nearest_distance = np.min(distances)
    duplicate = nearest_distance < 1e-06
    valid = correct_dimensions and within_bounds and (not duplicate)
    all_valid = all_valid and valid
for i in range(1, 9):
    formatted = ', '.join((f'{x:.6f}' for x in final_candidates[i]))

if not all_valid:
    raise ValueError("One or more final candidates failed validation")
for i in range(1, 9):
    print(f"F{i}: " + "-".join(f"{v:.6f}" for v in final_candidates[i]))

# Submission overview with methods and novelty distances.
method_labels = {1: "Space-filling exploration", 2: "NN-supported region",
                 3: "UCB (kappa=0.5)", 4: "Local refinement",
                 5: "Boundary refinement (min distance 0.03)",
                 6: "UCB (kappa=0.5)", 7: "Local refinement",
                 8: "UCB (kappa=0.5)"}
summary = []
for i in range(1, 9):
    X = data[i].drop(columns="output").to_numpy(dtype=float)
    point = final_candidates[i]
    summary.append({"Function": f"F{i}", "Method": method_labels[i],
                    "Nearest observation": float(np.min(cdist(point.reshape(1, -1), X))),
                    "Submission": "-".join(f"{v:.6f}" for v in point)})
display(pd.DataFrame(summary).round(6))

F1: 0.998294-0.000709
F2: 0.697103-0.994806
F3: 0.463879-0.497888-0.452731
F4: 0.386248-0.381010-0.343907-0.423092
F5: 1.000000-1.000000-0.970000-1.000000
F6: 0.426973-0.384648-0.468006-0.752822-0.084308
F7: 0.157801-0.305871-0.468629-0.230835-0.339422-0.619025
F8: 0.026226-0.080912-0.128250-0.056836-0.748250-0.399531-0.253527-0.720750


,Function,Method,Nearest observation,Submission
0,F1,Space-filling exploration,0.307658,0.998294-0.000709
1,F2,NN-supported region,0.068466,0.697103-0.994806
2,F3,UCB (kappa=0.5),0.075282,0.463879-0.497888-0.452731
3,F4,Local refinement,0.014092,0.386248-0.381010-0.343907-0.423092
4,F5,Boundary refinement (min distance 0.03),0.030000,1.000000-1.000000-0.970000-1.000000
5,F6,UCB (kappa=0.5),0.096193,0.426973-0.384648-0.468006-0.752822-0.084308
6,F7,Local refinement,0.047855,0.157801-0.305871-0.468629-0.230835-0.339422-0...
7,F8,UCB (kappa=0.5),0.280017,0.026226-0.080912-0.128250-0.056836-0.748250-0...
